# exp41 — GLM-OCR fine-tuned on Hebrew handwritten lines (exp35's data mix)

**Question:** can a pretrained document VLM, GLM-OCR (`zai-org/GLM-OCR`, 1.1B, MIT; trained on zh/en/fr/es/ru/de/ja/ko, **no Hebrew**),
be taught Hebrew handwriting with our data, and how does it compare with exp35-large (HHD validation CER greedy 0.0467, beam 12 + char LM
0.0398)? Its potential advantage over our CTC line models: it keeps Latin letters, digits and formulas (mixed pages), and it can later be
fine-tuned on whole pages.

**Same as exp35 / exp40:** all real lines (HHD train+test minus the same 10% validation writers, SCE with density correction, Pinkas) +
**20% synthetic digits/punctuation** per epoch (fresh draws, 1,000 held out), Mishkefet augmentation (0.7, ramped over 3 epochs),
checkpoint chosen on HHD validation (greedy CER).

**Added:** **5% English IAM lines** per epoch (keep Latin); IAM validation CER **before and after** fine-tuning = forgetting check.

**Model / format:** GLM-OCR's own prompt (`Text Recognition:`), the line image as is (RGB, **not mirrored**, 112 px high, at most
3,136 px wide), the transcription (logical order) as the answer + its end-of-turn token. Loss on the answer only. Hebrew is not in its
vocabulary (UTF-8 byte pieces, ~0.9 tokens per character), so the decoder learns Hebrew here. Patch embedding + lower 12 of the 24 vision
blocks frozen; AdamW, lr 3e-5 (language model + projector) / 1e-5 (upper vision blocks), warm-up 5% + linear decay, batch 8 x 2
accumulation, bf16 autocast, gradient checkpointing, up to 8 epochs, early stop after 3 epochs without a validation gain.
**Zero-shot** scores (before any training) are recorded first.

**A5000:** `./run.sh exp41/exp41_glmocr_line_allreal_synpunct_colab.ipynb` inside tmux, roughly 4–6 h (the per-epoch time is printed).
Output → public HF repo `cyttic/exp41-glmocr-line-allreal-synpunct` (model in bf16 + processor + results.json + train_log.tsv).

### 1. Setup

In [ ]:
!pip install -q -U "transformers==5.15.0" datasets jiwer opencv-python-headless safetensors

In [ ]:
import os, io, re, json, math, time, glob, random, pickle, types, shutil
import numpy as np, torch, jiwer
from PIL import Image
from datasets import load_dataset, concatenate_datasets
from huggingface_hub import snapshot_download, HfApi, login, whoami
import transformers
from transformers import AutoTokenizer, GlmOcrForConditionalGeneration
try:
    from transformers.models.glm46v.image_processing_pil_glm46v import Glm46VImageProcessorPil
except ImportError:
    from transformers import Glm46VImageProcessorPil
try:
    from google.colab import userdata; login(userdata.get("HF_TOKEN"))
except Exception:
    from huggingface_hub import get_token   # outside Colab (e.g. the A5000 server): the token saved by `hf auth login`
    if get_token() is None:
        from huggingface_hub import notebook_login; notebook_login()
print("HF user:", whoami()["name"], "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "NO GPU visible (nvidia-smi?) - refusing to train on CPU"
device = "cuda"
assert torch.cuda.get_device_capability()[0] >= 8, "needs bf16 (Ampere or newer)"
AMP_DTYPE = torch.bfloat16
print(device, torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")

### 2. Config

In [ ]:
BASE_ID  = "zai-org/GLM-OCR"                     # 1.1B, MIT
DATA_ID  = "cyttic/trocr-hebrew-matan"           # HHD: train 4,740 / test 1,455
MISH_ID  = "itayinbar/Mishkefet-v1"              # augmentation code only (no weights)
OUT_REPO = "cyttic/exp41-glmocr-line-allreal-synpunct"   # public
SCE_ID   = "cyttic/trocr-hebrew-human"           # SCE human set: 280 writers, 127 texts
PINKAS_ID        = "cyttic/pinkas-hebrew-lines"  # mirror of Zenodo 3569694 (CC-BY-4.0), line crops
PINKAS_LOCAL_DIR = "/mnt/ssd2/cyttic/datasets/pinkas_lines"   # used instead of HF when it exists
DENSITY_TARGET   = 22.9     # HHD train median px/char at 64 px height; SCE is compressed to it (as exp35)
HHF_ID        = "cyttic/heb-synth-punct-digits-hhf"                   # 20k synthetic lines
HHF_LOCAL_DIR = "/mnt/ssd2/cyttic/datasets/heb_synth_punct_digits_hhf"  # used instead of HF when it exists
IAM_ID        = "Teklia/IAM-line"                                     # real English lines (train / validation / test)
SYN_HOLDOUT   = 1000        # synthetic lines never trained on: final digit/punctuation check
SYN_FRAC, IAM_FRAC = 0.20, 0.05   # share of each epoch (real ~75%)
N_IAM_EVAL, N_SYN_EVAL_ZS = 300, 300   # IAM validation lines scored before/after; held-out synthetic lines for the zero-shot check
PUSH     = True

SEED            = 42
VAL_WRITER_FRAC = 0.10      # 10% of HHD writers -> validation (= exp35's split)
LINE_H, MAX_W   = 112, 3136 # line height fed to GLM-OCR (8 patches of 14 px); wider lines are squeezed, not cut
PROMPT_TEXT     = "Text Recognition:"            # GLM-OCR's own task prompt
FREEZE_VISION_BLOCKS = 12   # patch embedding + the lower 12 of 24 vision blocks frozen
LR_TEXT, LR_VISION = 3e-5, 1e-5
WARMUP, WD      = 0.05, 0.01
BATCH, ACCUM    = 8, 2      # effective batch 16 (= exp35)
EVAL_BATCH      = 24
EPOCHS, PATIENCE = 8, 3
MAX_TARGET_TOKENS = 256     # longer answers are cut (byte pieces: ~0.9 tokens per Hebrew character)
MAX_NEW_TOKENS  = 256
AUGMENT, AUG_MAX, AUG_RAMP = True, 0.7, 3   # Mishkefet's augmentation, strength ramped over 3 epochs (as exp35)
NUM_WORKERS     = min(8, os.cpu_count())
SMOKE           = False     # True = tiny run to test the pipeline end to end (nothing pushed)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT = "/content/exp41" if os.path.isdir("/content") else os.path.abspath("exp41_run")
os.makedirs(OUT, exist_ok=True); print("OUT", OUT)

### 3. Augmentation (Mishkefet, = exp35)

In [ ]:
MISH = snapshot_download(MISH_ID, allow_patterns=["hebocr/*", "hebocr/**"])
import sys; sys.path.insert(0, MISH)
from hebocr.data.transforms import Augment          # the augmentation only (exp35's)
augment = Augment(strength=AUG_MAX)

### 4. Data (= exp35 + 5% IAM) and line formatting (RGB, not mirrored, 112 px high)

In [ ]:
COLS = ["image", "text", "writer", "src"]
hhd = load_dataset(DATA_ID)
hhd_all = concatenate_datasets([hhd["train"], hhd["test"]])
hhd_all = hhd_all.map(lambda w: {"writer": "hhd:" + w, "src": "hhd"}, input_columns=["writer"]).select_columns(COLS)
sce = load_dataset(SCE_ID)
sce_all = concatenate_datasets([sce["train"], sce["test"]])
sce_all = sce_all.map(lambda d: {"writer": "sce:" + d, "src": "sce"}, input_columns=["source_doc"]).select_columns(COLS)
if os.path.isdir(PINKAS_LOCAL_DIR):
    pinkas = load_dataset("parquet", data_files={"train": sorted(glob.glob(os.path.join(PINKAS_LOCAL_DIR, "*.parquet")))}, split="train")
else:
    pinkas = load_dataset(PINKAS_ID, split="train")
pinkas = pinkas.map(lambda w: {"src": "pinkas"}, input_columns=["writer"]).select_columns(COLS)

wr = sorted(set(hhd_all["writer"])); random.Random(SEED).shuffle(wr)          # = exp35's validation writers
val_w = set(wr[:max(1, int(len(wr) * VAL_WRITER_FRAC))])
hw = hhd_all["writer"]
val = hhd_all.select([i for i, w in enumerate(hw) if w in val_w])
train = concatenate_datasets([hhd_all.select([i for i, w in enumerate(hw) if w not in val_w]), sce_all, pinkas]).select_columns(COLS)

if os.path.isdir(HHF_LOCAL_DIR):
    hhf = load_dataset("parquet", data_files={"train": sorted(glob.glob(os.path.join(HHF_LOCAL_DIR, "*.parquet")))}, split="train")
else:
    hhf = load_dataset(HHF_ID, split="train")
hhf = hhf.shuffle(seed=SEED)
syn_val = hhf.select(range(SYN_HOLDOUT))
syn = hhf.select(range(SYN_HOLDOUT, len(hhf)))
syn = syn.map(lambda st: {"writer": "synth:" + st, "src": "synth"}, input_columns=["style"]).select_columns(COLS)
syn_val = syn_val.map(lambda st: {"writer": "synth:" + st, "src": "synth"}, input_columns=["style"]).select_columns(COLS)

iam_tr = load_dataset(IAM_ID, split="train").shuffle(seed=SEED)
iam_tr = iam_tr.map(lambda t: {"writer": "iam", "src": "iam"}, input_columns=["text"]).select_columns(COLS)
iam_val = load_dataset(IAM_ID, split="validation").shuffle(seed=SEED)
iam_val = iam_val.select(range(min(N_IAM_EVAL, len(iam_val))))
iam_val = iam_val.map(lambda t: {"writer": "iam", "src": "iam"}, input_columns=["text"]).select_columns(COLS)

if SMOKE:
    train = concatenate_datasets([train.filter(lambda s_: s_ == src, input_columns=["src"]).select(range(16)) for src in ("hhd", "sce", "pinkas")])
    val, syn, syn_val = val.select(range(24)), syn.select(range(40)), syn_val.select(range(24))
    iam_tr, iam_val = iam_tr.select(range(16)), iam_val.select(range(16))
    EPOCHS, PATIENCE, N_SYN_EVAL_ZS = 1, 1, 24

def train_set(ep):
    """one epoch: all real + SYN_FRAC synthetic + IAM_FRAC English (fresh random draws per epoch; the synthetic draw = exp35/exp40's)"""
    total = len(train) / (1 - SYN_FRAC - IAM_FRAC)
    r = random.Random(SEED + ep)
    ps = r.sample(range(len(syn)), min(len(syn), int(total * SYN_FRAC)))
    pi = r.sample(range(len(iam_tr)), min(len(iam_tr), int(total * IAM_FRAC)))
    return concatenate_datasets([train, syn.select(ps), iam_tr.select(pi)])

from collections import Counter
src_lines = Counter(train["src"])
print("train lines by source:", dict(src_lines), "| epoch size:", len(train_set(1)))
print(f"train {len(train)} lines, {len(set(train['writer']))} writers | val {len(val)} lines ({len(val_w)} HHD writers) "
      f"| IAM eval {len(iam_val)} | synthetic held-out {len(syn_val)}")

def fmt(img):
    """line image -> RGB at LINE_H px height (NOT mirrored: GLM-OCR reads the page as it is), squeezed to <= MAX_W"""
    img = img.convert("RGB"); w, h = img.size
    nw = min(MAX_W, max(28, round(w * LINE_H / max(h, 1))))
    return img.resize((nw, LINE_H), Image.LANCZOS)

def prep(ex, train_mode):
    img = ex["image"]
    if ex.get("src") == "sce" and ex["text"].strip():          # SCE is 1.6x sparser than HHD (as exp35)
        dens = img.width * 64 / max(img.height, 1) / len(ex["text"])
        f = float(np.clip(DENSITY_TARGET / dens, 0.4, 8.0))
        if abs(f - 1) > 0.02:
            img = img.resize((max(16, int(round(img.width * f))), img.height), Image.BICUBIC)
    if train_mode and AUGMENT:
        img = Image.fromarray(augment.augment_gray(img))
    return fmt(img)

### 5. Model: GLM-OCR, its prompt format, answer-only loss

In [ ]:
class MiniProcessor:
    """GLM-OCR's processor for single-image prompts WITHOUT torchvision (the official Glm46VProcessor needs torchvision for its video part,
    which the server env does not have): the PIL image processor + the tokenizer; each <|image|> is expanded to grid_t*grid_h*grid_w/merge^2
    image tokens, exactly as the official processor does (checked locally: identical input_ids / attention_mask / mm_token_type_ids /
    image_grid_thw for left and right padding, pixel values within 3e-7)."""
    def __init__(self, repo):
        cfg = json.load(open(hf_hub_download(repo, "processor_config.json")))["image_processor"]
        self.image_processor = Glm46VImageProcessorPil(**{k: v for k, v in cfg.items() if k != "image_processor_type"})
        self.tokenizer = AutoTokenizer.from_pretrained(repo)
        self.merge = cfg["merge_size"]; self.image_token = "<|image|>"
        self.image_token_id = self.tokenizer.convert_tokens_to_ids(self.image_token)
    def __call__(self, text, images, padding_side="right"):
        im = self.image_processor(images=images, return_tensors="pt")
        grid = im["image_grid_thw"]; ids = []
        for t, g in zip(text, grid):
            assert t.count(self.image_token) == 1
            ids.append(self.tokenizer(t.replace(self.image_token, self.image_token * int(g.prod() // self.merge ** 2)), add_special_tokens=False)["input_ids"])
        L = max(map(len, ids)); pad = self.tokenizer.pad_token_id
        out_ids = torch.full((len(ids), L), pad, dtype=torch.long); att = torch.zeros((len(ids), L), dtype=torch.long)
        for i, x in enumerate(ids):
            sl = slice(L - len(x), L) if padding_side == "left" else slice(0, len(x))
            out_ids[i, sl] = torch.tensor(x); att[i, sl] = 1
        mm = (out_ids == self.image_token_id).long() * att      # text 0 / image 1: GLM's 3D (M-RoPE) positions need it
        return {"input_ids": out_ids, "attention_mask": att, "mm_token_type_ids": mm, "pixel_values": im["pixel_values"], "image_grid_thw": grid}

from huggingface_hub import hf_hub_download
processor = MiniProcessor(BASE_ID)
tok = processor.tokenizer
EOT = tok.convert_tokens_to_ids("<|user|>")          # GLM's end of turn (one of the generation eos ids)
THINK_END = tok.convert_tokens_to_ids("</think>")
NL = tok("\n", add_special_tokens=False)["input_ids"]; assert NL == [10], NL
# GLM-OCR's chat template rendered for one image + PROMPT_TEXT with the generation prompt, + "<think></think>\n" (= its training format)
PROMPT = "[gMASK]<sop><|user|>\n<|begin_of_image|><|image|><|end_of_image|>" + PROMPT_TEXT + "<|assistant|>\n<think></think>\n"
print(repr(PROMPT))

model = GlmOcrForConditionalGeneration.from_pretrained(BASE_ID, dtype=torch.float32).to(device)   # fp32 master weights, bf16 autocast
vis = model.model.visual
for p in vis.patch_embed.parameters(): p.requires_grad = False
for blk in vis.blocks[:FREEZE_VISION_BLOCKS]:
    for p in blk.parameters(): p.requires_grad = False
n_all = sum(p.numel() for p in model.parameters()); n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"GLM-OCR {n_all/1e6:.1f}M parameters, {n_tr/1e6:.1f}M trainable (patch embed + {FREEZE_VISION_BLOCKS} vision blocks frozen)")

def batch_train(exs, train_mode=True):
    """prompt (left context, image tokens expanded by the processor) + answer + <|user|>, right padded; labels = answer tokens only"""
    imgs = [prep(e, train_mode) for e in exs]
    ans = [tok(e["text"], add_special_tokens=False)["input_ids"][:MAX_TARGET_TOKENS] + [EOT] for e in exs]
    enc = processor([PROMPT] * len(exs), imgs, padding_side="right")
    plen = enc["attention_mask"].sum(1).tolist()
    L = max(p + len(a) for p, a in zip(plen, ans)); pad = tok.pad_token_id
    ids = torch.full((len(exs), L), pad, dtype=torch.long); att = torch.zeros((len(exs), L), dtype=torch.long)
    lab = torch.full((len(exs), L), -100, dtype=torch.long)
    has_mm = "mm_token_type_ids" in enc                                       # text 0 / image 1: GLM's 3D (M-RoPE) positions need it
    mm = torch.zeros((len(exs), L), dtype=enc["mm_token_type_ids"].dtype if has_mm else torch.long)
    for i, (p, a) in enumerate(zip(plen, ans)):
        ids[i, :p] = enc["input_ids"][i, :p]; ids[i, p:p + len(a)] = torch.tensor(a)
        if has_mm: mm[i, :p] = enc["mm_token_type_ids"][i, :p]
        att[i, :p + len(a)] = 1; lab[i, p:p + len(a)] = torch.tensor(a)
    out = {"input_ids": ids, "attention_mask": att, "labels": lab, "pixel_values": enc["pixel_values"], "image_grid_thw": enc["image_grid_thw"]}
    if has_mm: out["mm_token_type_ids"] = mm
    return out

def batch_gen(exs):
    imgs = [prep(e, False) for e in exs]
    enc = processor([PROMPT] * len(exs), imgs, padding_side="left")
    return enc, [e["text"] for e in exs]

class Collate:
    def __init__(self, fn, **kw): self.fn, self.kw = fn, kw
    def __call__(self, b): return self.fn(b, **self.kw)

def loader(split, fn, bs, shuffle, **kw):
    return torch.utils.data.DataLoader(split, batch_size=bs, shuffle=shuffle, num_workers=NUM_WORKERS, collate_fn=Collate(fn, **kw), drop_last=False)

# sanity check of the formatting on 2 real lines: the labels decode to exactly the transcription + <|user|>
b = batch_train([val[0], val[1]], train_mode=False)
for i in range(2):
    got = tok.decode([t for t in b["labels"][i].tolist() if t != -100])
    print(repr(got)); assert got == val[i]["text"] + "<|user|>", (got, val[i]["text"])
    p = int(b["attention_mask"][i].sum()) - int((b["labels"][i] != -100).sum())
    assert b["input_ids"][i, p - 1] == NL[0] and b["input_ids"][i, p - 2] == THINK_END       # answer starts right after "</think>\n"
print("input length", tuple(b["input_ids"].shape), "| image grid", b["image_grid_thw"].tolist())

### 6. Zero-shot scores, then fine-tuning (validation = HHD writers, greedy CER)

In [ ]:
_N = str.maketrans({"״": '"', "׳": "'"})
VARIANTS = {"raw": lambda t: t,
            "norm": lambda t: re.sub(r"\s+", " ", t.translate(_N)).strip(),
            "let": lambda t: re.sub(r" +", " ", re.sub(r"[^א-ת ]", " ", t)).strip()}
def scores(R, H, variants=VARIANTS):
    o = {}
    for v, f in variants.items():
        rr = [f(x) for x in R]; hh = [f(x) for x in H]; k = [i for i, x in enumerate(rr) if x]
        rr = [rr[i] for i in k]; hh = [hh[i] for i in k]
        o[f"cer_{v}"] = jiwer.cer(rr, hh); o[f"wer_{v}"] = jiwer.wer(rr, hh)
    return o
EN = {"raw": VARIANTS["raw"], "norm": VARIANTS["norm"]}        # IAM: no Hebrew-letter variant

@torch.no_grad()
def predict(split, num_beams=1):
    model.eval(); model.config.use_cache = True; H, R = [], []
    for enc, texts in loader(split, batch_gen, EVAL_BATCH, False):
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, num_beams=num_beams,
                                 eos_token_id=[EOT, tok.convert_tokens_to_ids("<|endoftext|>")], pad_token_id=tok.pad_token_id)
        H += [h.strip() for h in tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
        R += texts
    return R, H

def number_order(R, H):
    same = rev = other = 0
    for r_, h_ in zip(R, H):
        for run in re.findall(r"\d{2,}", r_):
            if run in h_: same += 1
            elif run[::-1] in h_: rev += 1
            else: other += 1
    return {"correct": same, "reversed": rev, "other": other}

def show(R, H, n=3):
    for i in range(min(n, len(R))): print("  REF", R[i]); print("  OUT", H[i])

# ---- zero-shot (before any training)
results = {"zero_shot": {}}
t0 = time.time()
R, H = predict(val); results["zero_shot"]["hhd_val"] = scores(R, H); print("zero-shot HHD val:", {k: round(v, 4) for k, v in results["zero_shot"]["hhd_val"].items()}); show(R, H)
R, H = predict(iam_val); results["zero_shot"]["iam_val"] = scores(R, H, EN); print("zero-shot IAM val:", {k: round(v, 4) for k, v in results["zero_shot"]["iam_val"].items()}); show(R, H)
sv = syn_val.select(range(min(N_SYN_EVAL_ZS, len(syn_val))))
R, H = predict(sv); results["zero_shot"]["syn_holdout_sample"] = scores(R, H); results["zero_shot"]["number_order"] = number_order(R, H)
print("zero-shot synthetic held-out:", {k: round(v, 4) for k, v in results["zero_shot"]["syn_holdout_sample"].items()}, results["zero_shot"]["number_order"]); show(R, H)
print(f"zero-shot evaluation {time.time()-t0:.0f}s", flush=True)

# ---- fine-tuning
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
vis_ids = {id(p) for p in vis.parameters()}
groups = [{"params": [p for p in model.parameters() if p.requires_grad and id(p) in vis_ids], "lr": LR_VISION},
          {"params": [p for p in model.parameters() if p.requires_grad and id(p) not in vis_ids], "lr": LR_TEXT}]
opt = torch.optim.AdamW(groups, weight_decay=WD, fused=True); base = [g["lr"] for g in opt.param_groups]
steps_per_ep = math.ceil(math.ceil(len(train_set(1)) / BATCH) / ACCUM)
total = EPOCHS * steps_per_ep; warm = max(1, int(WARMUP * total)); step = 0
LOG = os.path.join(OUT, "train_log.tsv")
open(LOG, "w").write("epoch\tstep\ttrain_loss\tval_cer_raw\tval_wer_raw\tval_empty_frac\tsec\n")
best = {"cer": 9.9, "state": None, "epoch": None}; since_best = 0; T0 = time.time()
for ep in range(1, EPOCHS + 1):
    augment.strength = AUG_MAX * min(1.0, 0.3 + 0.7 * (ep - 1) / max(1, AUG_RAMP - 1))
    model.train(); model.config.use_cache = False; t0 = time.time(); losses = []; opt.zero_grad(set_to_none=True)
    for it, b in enumerate(loader(train_set(ep), batch_train, BATCH, True, train_mode=True)):
        b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            loss = model(**b).loss
        (loss / ACCUM).backward(); losses.append(float(loss))
        if (it + 1) % ACCUM == 0:
            f = min(1.0, (step + 1) / warm) * max(0.0, (total - step) / max(1, total - warm))   # warm-up + linear decay
            for g, b0 in zip(opt.param_groups, base): g["lr"] = b0 * f
            torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], 1.0)
            opt.step(); opt.zero_grad(set_to_none=True); step += 1
        if it % 200 == 0:
            print(f"  ep{ep} it{it} loss {np.mean(losses[-200:]):.4f} | {time.time()-t0:.0f}s | max mem {torch.cuda.max_memory_allocated()/2**30:.1f} GB", flush=True)
    tr_s = time.time() - t0
    R, H = predict(val); s = scores(R, H); empty = float(np.mean([len(h) == 0 for h in H]))
    open(LOG, "a").write(f"{ep}\t{step}\t{np.mean(losses):.4f}\t{s['cer_raw']:.4f}\t{s['wer_raw']:.4f}\t{empty:.3f}\t{time.time()-t0:.0f}\n")
    print(f"[ep{ep}] loss {np.mean(losses):.4f} | HHD val CER {s['cer_raw']:.4f} WER {s['wer_raw']:.4f} | empty {empty:.3f} | aug {augment.strength:.2f} "
          f"| train {tr_s/60:.1f} min, eval {(time.time()-t0-tr_s)/60:.1f} min | total {(time.time()-T0)/3600:.2f} h", flush=True); show(R, H, 2)
    if s["cer_raw"] < best["cer"]:
        best.update(cer=s["cer_raw"], epoch=ep, state={k: v.detach().to("cpu", torch.bfloat16).clone() for k, v in model.state_dict().items()}); since_best = 0
    else:
        since_best += 1
        if since_best >= PATIENCE: print(f"early stop: no validation gain for {PATIENCE} epochs", flush=True); break
TRAIN_HOURS = (time.time() - T0) / 3600
model.load_state_dict(best["state"]); del opt; torch.cuda.empty_cache()
print("best checkpoint: epoch", best["epoch"], "| HHD val CER", round(best["cer"], 4), f"| {TRAIN_HOURS:.2f} h")

### 7. Final checks — HHD validation, held-out synthetic (digits / punctuation / number order), IAM (forgetting)

In [ ]:
results["fine_tuned"] = {}
for name, nb in (("greedy", 1), ("beam4", 4)):
    r = {}
    R, H = predict(val, nb); r["hhd_val"] = scores(R, H)
    Rs, Hs = predict(syn_val, nb); r["syn_holdout"] = scores(Rs, Hs); r["number_order"] = number_order(Rs, Hs)
    Ri, Hi = predict(iam_val, nb); r["iam_val"] = scores(Ri, Hi, EN)
    results["fine_tuned"][name] = r
    print(f"{name:7s} HHD val CER {r['hhd_val']['cer_raw']:.4f} WER {r['hhd_val']['wer_raw']:.4f} | synthetic held-out CER "
          f"{r['syn_holdout']['cer_raw']:.4f} | IAM val CER {r['iam_val']['cer_raw']:.4f} | numbers {r['number_order']}")
print(f"\nexp35-large on the same HHD validation writers: greedy 0.0467, beam 12 + char LM 0.0398")
print(f"IAM val CER: zero-shot {results['zero_shot']['iam_val']['cer_raw']:.4f} -> fine-tuned {results['fine_tuned']['greedy']['iam_val']['cer_raw']:.4f}")
print("\nHHD val examples:"); show(R, H, 5)
print("\nsynthetic held-out examples:"); show(Rs, Hs, 3)
print("\nIAM examples:"); show(Ri, Hi, 3)

### 8. Save to the public HF repo

In [ ]:
MD = os.path.join(OUT, "model"); os.makedirs(MD, exist_ok=True)
model.to(torch.bfloat16).save_pretrained(MD); tok.save_pretrained(MD)
for f in ("processor_config.json", "preprocessor_config.json", "chat_template.jinja"):   # the base repo's processor files, unchanged
    shutil.copy(hf_hub_download(BASE_ID, f), os.path.join(MD, f))
json.dump({"arch": "GLM-OCR (zai-org/GLM-OCR) fine-tuned on Hebrew handwritten lines (exp41)", "base": BASE_ID,
           "format": {"prompt": PROMPT, "answer_end": "<|user|>", "line_height": LINE_H, "max_width": MAX_W, "mirrored": False,
                      "note": "generate from PROMPT (chat template + '<think></think>\\n') with the line image; stop at <|user|>"},
           "sources": {"hhd": DATA_ID + " train+test (minus validation writers)", "sce": SCE_ID + " train+test", "pinkas": PINKAS_ID,
                       "synthetic": HHF_ID + f" (minus {SYN_HOLDOUT} held out)", "iam": IAM_ID + " train"},
           "mix": {"synthetic_frac": SYN_FRAC, "iam_frac": IAM_FRAC}, "train_lines": dict(src_lines), "density_target": DENSITY_TARGET,
           "train": {"lr_text": LR_TEXT, "lr_vision": LR_VISION, "frozen": f"patch embed + vision blocks 0-{FREEZE_VISION_BLOCKS - 1}",
                     "wd": WD, "warmup": WARMUP, "batch": BATCH, "accum": ACCUM, "epochs_max": EPOCHS, "patience": PATIENCE,
                     "schedule": "linear warm-up + linear decay", "augment": f"Mishkefet Augment {AUG_MAX}, ramp {AUG_RAMP}",
                     "precision": "fp32 master weights, bf16 autocast", "seed": SEED, "hours": TRAIN_HOURS},
           "best_epoch": best["epoch"], "val_cer_raw": best["cer"],
           "compare_with": {"cyttic/exp35-trocr-large-en-ctc-seqhead-allreal-synpunct": {"hhd_val_greedy": 0.0467, "hhd_val_beam12_charLM": 0.0398}},
           "note": "trained on HHD train+test, SCE, Pinkas: evaluate on the ivrit.ai benchmark only",
           "results": results}, open(os.path.join(OUT, "results.json"), "w"), indent=1)
ft = results["fine_tuned"]["greedy"]
open(os.path.join(OUT, "README.md"), "w").write(
    "---\nlicense: mit\nlanguage: [he, en]\nbase_model: zai-org/GLM-OCR\ntags: [ocr, htr, hebrew, handwriting]\n---\n"
    "# exp41 — GLM-OCR fine-tuned on Hebrew handwritten lines\n\n"
    "zai-org/GLM-OCR fine-tuned on HHD + SCE + Pinkas + 20% synthetic digits/punctuation + 5% English IAM lines (line level). "
    f"Input: the line image (not mirrored) at {LINE_H} px height; prompt `Text Recognition:` in GLM-OCR's chat format followed by "
    "`<think></think>\\n`; the answer ends with `<|user|>`. Weights in `model/` (bf16, with the processor).\n\n"
    f"HHD validation CER (greedy): {ft['hhd_val']['cer_raw']:.4f} | IAM validation CER: {ft['iam_val']['cer_raw']:.4f} "
    f"(zero-shot {results['zero_shot']['iam_val']['cer_raw']:.4f}). Details in results.json.\n")
if PUSH and not SMOKE:
    api = HfApi(); api.create_repo(OUT_REPO, private=False, exist_ok=True)
    for k in range(5):
        try: api.upload_folder(repo_id=OUT_REPO, folder_path=OUT, commit_message="exp41 GLM-OCR line fine-tune"); break
        except Exception as e: print("upload failed, retrying:", e); time.sleep(30)
    print("pushed -> https://huggingface.co/" + OUT_REPO)
else:
    print("not pushed (SMOKE or PUSH=False); files in", OUT)